## Run these cells until you reach the **Execution Note**.

## **Login to HuggingFace Hub**

This cell authenticates the session with the HuggingFace Hub using an access token, allowing gated models and private repositories to be downloaded.

In [ ]:
from huggingface_hub import login
login('')

## **Define Fixed Dependency Constraints**

This cell creates a `constraints.txt` file with pinned versions of core scientific and PyTorch libraries to ensure reproducible training and avoid compatibility issues.

In [ ]:
# Write constraints file
with open("/kaggle/working/constraints.txt", "w") as f:
    f.write("""numpy==1.26.4
scipy==1.11.4
lightning==2.2.5
torch==2.3.1+cu121
torchaudio==2.3.1+cu121
torchvision==0.18.1+cu121
""")

## **Install PyTorch and Pyannote Stack with Constraints**

This cell installs NumPy, SciPy, the CUDA-enabled PyTorch stack, Lightning, and `pyannote.audio` using the predefined constraints file to maintain version compatibility and ensure stable diarization training.

In [ ]:
# Install NumPy/SciPy
!pip install --no-cache-dir --force-reinstall -c /kaggle/working/constraints.txt numpy scipy

# Install torch stack
!pip install -c /kaggle/working/constraints.txt \
  torch==2.3.1+cu121 torchaudio==2.3.1+cu121 torchvision==0.18.1+cu121 \
  --index-url https://download.pytorch.org/whl/cu121

# Install lightning + pyannote under constraints
!pip install -c /kaggle/working/constraints.txt lightning==2.2.5
!pip install -c /kaggle/working/constraints.txt pyannote.audio==3.1.1

## **Clone and Install Diarizers (No Dependency Overrides)**

This cell clones the HuggingFace `diarizers` repository into a separate directory and installs it in editable mode without modifying existing dependencies.

In [ ]:
!git clone https://github.com/huggingface/diarizers.git hf_diarizers
!pip install --no-deps -e /kaggle/working/hf_diarizers

## **Install Audio Augmentation and Denoising Libraries**

This cell installs `audiomentations` and `denoiser` under the defined constraints to support data augmentation and optional speech enhancement during diarization training.

In [ ]:
!pip install -c /kaggle/working/constraints.txt audiomentations

In [ ]:
!pip install -c /kaggle/working/constraints.txt denoiser

## **Install Demucs and Fix Hydra Dependencies**

This cell installs `demucs` under the defined constraints and reinstalls compatible versions of `omegaconf` and `hydra-core` to preserve the expected module structure used by many audio and diarization repositories.

In [ ]:
!pip install -c /kaggle/working/constraints.txt demucs

In [ ]:
!pip uninstall -y omegaconf hydra-core

# Install versions that keep the old module layout expected by many audio repos
!pip install -c /kaggle/working/constraints.txt --no-cache-dir --force-reinstall "omegaconf==2.3.0" "hydra-core==1.3.2"

# **Execution Note**

## Click **'Restart & Clear Cell Outputs'** and then start executing **only from the cells below**. **Do not** rerun the previous cells. .

In [ ]:
import diarizers
from diarizers.models import SegmentationModel

## **Verify Diarizers Installation**

This cell confirms that the `diarizers` package is correctly installed and imports the `SegmentationModel` to ensure the training environment is properly configured.

In [ ]:
import diarizers
print("Loaded from:", diarizers.__file__)

from diarizers.models import SegmentationModel
print("OK:", SegmentationModel)

## **Setup Imports and Paths for Inference**

This cell loads the required libraries and defines the test audio directory, fine-tuned model path, and output CSV filename for running diarization inference.

In [ ]:
import os
import sys
import json
import shutil
import pathlib
import subprocess
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm
import subprocess, sys, pathlib

from pyannote.audio import Pipeline
from pyannote.audio.pipelines.utils.hook import ProgressHook

import torch
from torch.torch_version import TorchVersion


TEST_DIR = "/kaggle/input/competitions/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio"
MODEL_PATH = "/kaggle/input/models/arpitamallik23/823-olt-diarization-final-model/pytorch/default/1"
OUTPUT_CSV = "diarization_final_submission.csv"

## **Demucs Vocal Separation Utility**

This cell defines a helper function that runs Demucs (`htdemucs`) to extract the vocal stem from an input audio file, includes detailed error logging for debugging, and returns the path to the separated vocals file for downstream diarization inference.

In [ ]:
# -------------------------
# Demucs
# -------------------------

demucs_out_dir = "/kaggle/working/demucs_out"
os.makedirs(demucs_out_dir, exist_ok=True)


def demucs_two_stem_vocals(in_wav_path, demucs_out_dir=demucs_out_dir):
    os.makedirs(demucs_out_dir, exist_ok=True)

    cmd = [
        sys.executable, "-m", "demucs",
        "-n", "htdemucs",
        "--two-stems", "vocals",
        "-o", demucs_out_dir,
        "--filename", "{track}/{stem}.{ext}", 
        in_wav_path
    ]

    # capture output so we can print error details
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

    if p.returncode != 0:
        print("\nDEMUCS FAILED:", in_wav_path)
        print("CMD:", " ".join(cmd))
        print("\n--- STDOUT (tail) ---\n", p.stdout[-3000:])
        print("\n--- STDERR (tail) ---\n", p.stderr[-6000:])
        raise RuntimeError("Demucs failed (see logs above).")

    track_name = pathlib.Path(in_wav_path).stem
    vocals_path = f"{demucs_out_dir}/htdemucs/{track_name}/vocals.wav"

    if not os.path.exists(vocals_path):
        # sometimes demucs naming differs; list what it produced
        produced_dir = f"{demucs_out_dir}/htdemucs/{track_name}"
        print("Expected vocals not found. Produced files:", produced_dir)
        if os.path.exists(produced_dir):
            print(os.listdir(produced_dir))
        raise FileNotFoundError(f"Missing expected output: {vocals_path}")

    return vocals_path

## **Run Diarization Inference and Generate Submission**

This cell loads the diarization pipeline, replaces its segmentation model with the fine-tuned checkpoint, runs inference over all test audio (optionally using Demucs vocal separation), and writes the final diarization results to the submission CSV.

### remember to turn on the GPU.

In [ ]:
# -------------------------
# Make diarizers visible
# -------------------------
SRC = "/kaggle/working/diarizers/src"
if SRC not in sys.path:
    sys.path.insert(0, SRC)

for k in list(sys.modules.keys()):
    if k == "diarizers" or k.startswith("diarizers."):
        del sys.modules[k]

from diarizers.models import SegmentationModel

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

# -------------------------
# Load pipeline
# -------------------------
pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")

pipeline.to(device)
print("Pipeline loaded successfully on", device)

ft_model = SegmentationModel.from_pretrained(MODEL_PATH)
ft_model = ft_model.to_pyannote_model().to(device)
pipeline._segmentation.model = ft_model

results = []

for filename in tqdm(sorted(os.listdir(TEST_DIR))):
    if not filename.endswith(".wav"):
        continue

    file_path = os.path.join(TEST_DIR, filename)
    file_id = os.path.splitext(filename)[0]

    try:
        vocals_path = demucs_two_stem_vocals(file_path)

        waveform, sample_rate = torchaudio.load(vocals_path)
        waveform = waveform.to(device)

        with ProgressHook() as hook:
            output = pipeline({"waveform": waveform, "sample_rate": sample_rate}, hook=hook)

        annotation = output

        segments = []
        for seg, _, speaker in annotation.itertracks(yield_label=True):
            segments.append({
                "start_time": int(round(seg.start)),
                "end_time": int(round(seg.end)),
                "speaker_id": speaker
            })

        results.append({"filename": file_id, "diarization": json.dumps(segments)})

    finally:
        track_folder = f"{demucs_out_dir}/htdemucs/{file_id}"
        if os.path.exists(track_folder):
            shutil.rmtree(track_folder, ignore_errors=True)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# -------------------------
# Submission
# -------------------------
df_submission = pd.DataFrame(results)
df_submission.to_csv(OUTPUT_CSV, index=False)
print("Saved:", OUTPUT_CSV)
df_submission.head()